Code used to document Augsburg City Forest Exploration.

In [1]:
import osmnx as ox
import gpxpy
import matplotlib.pyplot as plt
ox.settings.use_cache = True
import folium
import math
from shapely.geometry import LineString, MultiLineString
from shapely.ops import unary_union
import os

Trying folium for dynamic view

In [9]:
green_files = [
    os.path.join(r'C:\Users\schmueha\Documents\GitHub\hannaschmueck.github.io\datasets\Augsburg_Routes\Done', f)
    for f in os.listdir(r'C:\Users\schmueha\Documents\GitHub\hannaschmueck.github.io\datasets\Augsburg_Routes\Done')
    if f.endswith('.gpx')
]

In [10]:
# Exclusion zone
EX_LEFT, EX_BOTTOM, EX_RIGHT, EX_TOP = 48.307788, 10.914767, 48.308446, 10.916383

def point_in_exclusion(lat, lon):
    return EX_LEFT <= lat <= EX_RIGHT and EX_BOTTOM <= lon <= EX_TOP

def split_outside_exclusion(points):
    """Split a list of points into segments that avoid the exclusion zone."""
    segments = []
    current = []
    for lat, lon in points:
        if point_in_exclusion(lat, lon):
            if len(current) > 1:
                segments.append(current)
            current = []
        else:
            current.append((lat, lon))
    if len(current) > 1:
        segments.append(current)
    return segments

left, bottom, right, top = 48.271104, 10.891647, 48.356372, 10.946235
m = folium.Map(location=[(left + right) / 2, (bottom + top) / 2], zoom_start=13, tiles='CartoDB positron')

gpx_files = [('stadtwald.gpx', '#aaaaaa')] + [(f, '#27ae60') for f in green_files]

for gpx_file, color in gpx_files:
    with open(gpx_file, 'r') as f:
        gpx = gpxpy.parse(f)
        for track in gpx.tracks:
            for segment in track.segments:
                points = [(p.latitude, p.longitude) for p in segment.points]
                for clean_segment in split_outside_exclusion(points):
                    folium.PolyLine(
                        clean_segment, color=color,
                        weight=2 if color == '#aaaaaa' else 3,
                        opacity=0.5 if color == '#aaaaaa' else 0.8,
                        tooltip=track.name or gpx_file
                    ).add_to(m)

m.save('cyclemap.html')

In [11]:


BUFFER_M = 3

def gpx_to_linestring(path):
    with open(path) as f:
        gpx = gpxpy.parse(f)
    pts = []
    for track in gpx.tracks:
        for seg in track.segments:
            pts += [(p.longitude, p.latitude) for p in seg.points]
    return LineString(pts) if len(pts) > 1 else None

def gpx_to_lines(path):
    lines = []
    with open(path) as f:
        gpx = gpxpy.parse(f)
    for track in gpx.tracks:
        for seg in track.segments:
            pts = [(p.longitude, p.latitude) for p in seg.points]
            if len(pts) > 1:
                lines.append(LineString(pts))
    return lines

# Reference network
grid_lines = gpx_to_lines('stadtwald.gpx')
grid_ml = MultiLineString(grid_lines)

deg_per_m = 1 / (111320 * math.cos(math.radians(48.37)))
buf = BUFFER_M * deg_per_m

# Green routes — union the buffers first so overlaps aren't double-counted
green_lines = [gpx_to_linestring(f) for f in green_files if gpx_to_linestring(f)]
green_union = unary_union([l.buffer(buf) for l in green_lines])  # single merged polygon

# Clip the entire reference network against the merged green area
# This gives exact partial coverage — a segment half-covered counts as half
covered_geometry = grid_ml.intersection(green_union)
covered = covered_geometry.length if not covered_geometry.is_empty else 0

grid_total = grid_ml.length
pct = covered / grid_total * 100

print(f"Grid covered: {pct:.1f}%  ({covered * 111.320:.2f} km of {grid_total * 111.320:.2f} km)")

Grid covered: 12.3%  (35.48 km of 287.69 km)
